<div style="border-left: 5px solid #b7791f; background-color: #fff8e1; padding: 0.8em 1em; margin: 1em 0; border-radius: 4px;">
  <strong>Warning: AI-assisted materials</strong><br><br>
  These materials were developed with assistance from AI tools. All content has been reviewed and edited by the instructor, who takes final responsibility for its accuracy, clarity, and appropriateness for the course. Students should treat these materials as instructor-reviewed course content while applying the same critical judgment they would use with any technical material. Please report any suspected errors or unclear explanations to ghunt@wm.edu.
</div>

# Nearest-Neighbor Methods

Nearest-neighbor methods predict from nearby training observations: average their responses for regression, or have their labels vote for classification. We will first derive the population targets that these local averages estimate.

## Population Risk and the Optimal Score Function

Recall the score/action framework
$$
f(x)=a(s(x)).
$$
The score function $s$ produces the numerical quantity used by the action $a$ to make a prediction. In regression, $a$ is the identity. In classification, $a$ selects the class with the largest score.

For a new observation $(X,Y)\sim P$, the population risk of a score function $s$ is
$$
R(s)=\mathbb E[\ell(Y,s(X))].
$$
The **population-optimal score function** is
$$
s^*=\arg\min_s R(s).
$$


## Pointwise Minimization

At a fixed input $x$, define the **conditional risk** of proposing the score $t$:
$$
L_x(t)=\mathbb E[\ell(Y,t)\mid X=x].
$$
The law of total expectation rewrites the overall risk as
$$
R(s)=\mathbb E_X\big[L_X(s(X))\big].
$$
This says that overall risk averages the conditional risk incurred at each possible input.

The important additional assumption is that we are minimizing over **all possible score functions**. Then the values $s(x)$ and $s(x')$ may be chosen independently when $x\ne x'$.

To see the logic concretely, suppose $X$ takes values $x_1,\ldots,x_J$, with $p_j=\mathbb P(X=x_j)$. Then
$$
R(s)=\sum_{j=1}^J p_j L_{x_j}(s(x_j)) = p_1 L_{x_1}(s(x_1)) + p_2 L_{x_2}(s(x_2)) +  \cdots p_J L_{x_J}(s(x_J))
$$
The value $s(x_j)$ occurs only in the $j$th term. Changing $s(x_j)$ cannot change any other term, and all $p_j$ are nonnegative. We therefore make the whole sum as small as possible by choosing each $s(x_j)$ to minimize its own term:
$$
s^*(x_j)= \arg\min_t L_{x_j}(t).
$$
For continuous $X$, the sum becomes an integral over all $x$, but the reasoning is the same. At each input,
$$
\boxed{s^*(x)=\arg\min_t L_{x_j}(t)=\arg\min_t\mathbb E[\ell(Y,t)\mid X=x].}
$$

This pointwise argument would not apply directly if we restricted $s$ to a linked family such as $s(x)=w^\top x$. Changing $w$ changes the scores at many inputs at once, so improving the conditional risk at one input may worsen it elsewhere.

## Regression with Squared Loss

For regression, the score itself is the prediction, and squared loss is
$$
\ell(y,t)=(y-t)^2.
$$
Fix an input $x$. The proposed prediction $t$ is now an ordinary real number, so
$$
s^*(x)=\arg\min_{t\in\mathbb R}\mathbb E[(Y-t)^2\mid X=x].
$$
Expanding the square and taking the conditional expectation gives
$$
\begin{aligned}
L_x(t)
&=\mathbb E[Y^2-2tY+t^2\mid X=x]\\
&=\mathbb E[Y^2\mid X=x]-2t\mathbb E[Y\mid X=x]+t^2.
\end{aligned}
$$
Here the conditional moments are constants with respect to $t$. Therefore
$$
L_x'(t)=-2\mathbb E[Y\mid X=x]+2t.
$$
Setting the derivative equal to zero gives
$$
t=\mathbb E[Y\mid X=x].
$$
Because $L_x''(t)=2>0$, this critical point is the unique minimum. Hence
$$
\boxed{s^*(x)=\mathbb E[Y\mid X=x].}
$$

## From the Conditional Mean to Nearest-Neighbor Regression

The result
$$
s^*(x)=\mu(x)=\mathbb E[Y\mid X=x]
$$
identifies the population target, but it does not yet give us a fitted predictor: we do not know $P$, so we do not know $\mu(x)$.

Suppose we had many training observations whose input was exactly $x$. Their responses would come from the conditional distribution $Y\mid X=x$, so their average would estimate $\mu(x)$. With continuous features, however, exact matches are rare. We usually have no observations satisfying $x_n=x$.

### The Local Approximation

Nearest-neighbor regression replaces exact matches with nearby inputs. Its working assumption is that the conditional mean changes gradually in the chosen distance: when $x_n$ is close to $x$,
$$
\mu(x_n)\approx\mu(x).
$$
Alternatively, we are approximating
$$
\mathbb E[Y\mid X=x] \approx \text{ mean of $x_n$ near $x$.}
$$

This method is conventionally called **$k$-nearest neighbors**, or **KNN**, which explains software names such as `KNeighborsRegressor`. We instead use $m$ for the **number of neighbors**, leaving $K$ for the number of classes. Let $\mathcal N_m(x)$ contain the indices of the $m$ nearest training inputs to $x$. The fitted $m$-nearest-neighbor regression rule is
$$
\boxed{\hat s_m(x)=\frac1m\sum_{n\in\mathcal N_m(x)}y_n,
\qquad \hat f_m(x)=\hat s_m(x).}
$$

For a query $x$, the fitted procedure:

1. computes its distance to every training input;
2. selects the $m$ nearest, using a fixed rule for distance ties;
3. averages their responses.

The approximation involves a tradeoff. A very small $m$ keeps the neighborhood local but averages little noise. A large $m$ averages more responses but may include points whose conditional means differ substantially from $\mu(x)$. Distance, feature scaling, and $m$ together determine what “local” means.

Linear regression and nearest-neighbor regression estimate the same conditional mean. Linear regression uses a global form such as $w^\top x$; nearest-neighbor regression uses a local average. Its stored training observations are part of the fitted predictor.

## Regression Example: Flexibility

The data below follow $Y=\sin(X)+\varepsilon$. We keep the same observations while changing $m$.


In [ ]:
#| code-fold: true
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None

rng = np.random.default_rng(7)
X_reg = np.sort(rng.uniform(0, 2*np.pi, 100)).reshape(-1, 1)
y_reg = np.sin(X_reg[:, 0]) + rng.normal(0, 0.3, len(X_reg))
X_grid_reg = np.linspace(0, 2*np.pi, 300).reshape(-1, 1)

def draw_regression(ax, m):
    model = KNeighborsRegressor(n_neighbors=m).fit(X_reg, y_reg)
    ax.scatter(X_reg[:, 0], y_reg, s=12, alpha=0.5, label='training data')
    ax.plot(X_grid_reg[:, 0], np.sin(X_grid_reg[:, 0]), color='black', label='conditional mean')
    ax.plot(X_grid_reg[:, 0], model.predict(X_grid_reg), label='nearest-neighbor prediction')
    ax.set(title=f'm = {m}', xlabel='x', ylabel='y', ylim=(-1.8, 1.8))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, m in zip(axes, [1, 10, 100]):
    draw_regression(ax, m)
axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

def plot_nearest_neighbor_regression(m=10):
    fig, ax = plt.subplots(figsize=(7, 3.5))
    draw_regression(ax, m)
    ax.legend()
    plt.show()

if widgets is not None:
    display(widgets.interactive(
        plot_nearest_neighbor_regression,
        m=widgets.IntSlider(min=1, max=100, value=10, continuous_update=False),
    ))


**Smaller $m$ means greater flexibility**, reversing the direction we saw for polynomial degree.

- At $m=1$, predictions follow individual responses, including their noise. With distinct training inputs, training error is zero.
- Moderate $m$ averages away some noise while retaining local structure.
- At $m=N$, every prediction is the overall training-response mean: local structure disappears.

A larger neighborhood can reduce sensitivity to individual observations, but may average together inputs with different conditional means.

### Selecting $m$: Train, Validation, and Test

Use one three-way split: fit candidates on training data, choose $m$ by validation RMSE, refit on training plus validation, then evaluate on the test set. This is the same model-building procedure as in the previous lecture.


In [ ]:
rng = np.random.default_rng(70)
X = rng.uniform(0, 2*np.pi, (300, 1))
y = np.sin(X[:, 0]) + rng.normal(0, 0.3, len(X))
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=70)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=71)

m_grid = np.arange(1, 50)
val_rmse = []
for m in m_grid:
    model = KNeighborsRegressor(n_neighbors=int(m)).fit(X_train, y_train)
    val_rmse.append(np.sqrt(mean_squared_error(y_val, model.predict(X_val))))

best_m = int(m_grid[np.argmin(val_rmse)])
plt.plot(m_grid, val_rmse)
plt.scatter(best_m, min(val_rmse), color='crimson', label=f'Selected m = {best_m}')
plt.xlabel('Number of neighbors m')
plt.ylabel('Validation RMSE')
plt.legend()
plt.show()

final_model = KNeighborsRegressor(n_neighbors=best_m).fit(X_trainval, y_trainval)
test_rmse = np.sqrt(mean_squared_error(y_test, final_model.predict(X_test)))
print(f'Selected m: {best_m}; final test RMSE: {test_rmse:.3f}')


The winning validation score helped choose $m$, so it is not an independent final evaluation. The test score evaluates the refitted output of the entire selection procedure. CV could supply the candidate performance estimates instead; once used to select $m$, the winning CV score has the same selection issue.

## Classification and the Bayes Decision

Now suppose $Y\in\{1,\ldots,K\}$, where $K$ is the number of classes. The score function produces a vector
$$
s(x)=(s_1(x),\ldots,s_K(x))^\top,
$$
and the action predicts the class with the largest score:
$$
a(s(x))=\arg\max_{1\le c\le K}s_c(x).
$$

Under zero-one loss,
$$
\ell(y,s(x))=\mathbf1\{y\ne a(s(x))\}.
$$
At a fixed input $x$, the conditional risk of a proposed score vector $t$ is
$$
\begin{aligned}
L_x(t)
&=\mathbb E[\mathbf1\{Y\ne a(t)\}\mid X=x]\\
&=\mathbb P(Y\ne a(t)\mid X=x)\\
&=1-\mathbb P(Y=a(t)\mid X=x).
\end{aligned}
$$
The loss depends on $t$ only through the class $a(t)$ that it selects. Therefore minimizing conditional error is equivalent to choosing a class with the largest conditional probability:
$$
\boxed{f^*(x)=\arg\max_{1\le c\le K}\mathbb P(Y=c\mid X=x).}
$$
This is the **Bayes classifier**. Unlike squared-loss regression, the optimal score vector is not unique under zero-one loss: any scores whose largest component identifies a most probable class give the same decision. One convenient population score is the vector of conditional class probabilities
$$
s_c^*(x)=\eta_c(x)=\mathbb P(Y=c\mid X=x).
$$

### Nearest-Neighbor Classification

The probabilities $\eta_c(x)$ are unknown. As in regression, the nearest-neighbor method assumes that nearby inputs have similar conditional behavior:
$$
x_n\approx x
\quad\Longrightarrow\quad
\eta_c(x_n)\approx\eta_c(x).
$$
For class $c$, the indicator $\mathbf1\{Y=c\}$ has conditional mean
$$
\mathbb E[\mathbf1\{Y=c\}\mid X=x]=\mathbb P(Y=c\mid X=x)=\eta_c(x).
$$
We can therefore estimate $\eta_c(x)$ by averaging these indicators among the $m$ nearest neighbors:
$$
\hat s_{m,c}(x)=\hat\eta_{m,c}(x)
=\frac1m\sum_{n\in\mathcal N_m(x)}\mathbf1\{y_n=c\}.
$$
This is simply the fraction of the neighbors belonging to class $c$. The estimated probabilities are nonnegative and sum to one. Applying the usual action gives
$$
\boxed{
\hat f_m(x)=a(\hat s_m(x))
=\arg\max_{1\le c\le K}\hat\eta_{m,c}(x).
}
$$
Thus nearest-neighbor classification estimates the conditional class probabilities locally and predicts the most common neighbor label. A fixed rule is still needed if the class counts tie.

Multiclass logistic regression estimates probabilities through linear scores and softmax. Nearest-neighbor classification estimates probabilities by local class proportions. Both ultimately select the class with the largest estimated probability, but they obtain those estimates in different ways.

## Penguins: Boundaries and Distance

Use bill length and flipper length to predict species. The plots illustrate fitted decision regions using all displayed observations; they are not performance estimates.

Euclidean distance is
$$
d(x,x')=\sqrt{\sum_{j=1}^D(x_j-x'_j)^2}.
$$
A feature with a larger numerical scale can dominate this sum. Standardization changes the distance so that features are measured in standard-deviation units. This can be useful, but is a modeling choice rather than a guarantee of better predictions. During evaluation, learn the scaler from training data only.

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from matplotlib.colors import ListedColormap

url = 'https://gist.githubusercontent.com/slopp/ce3b90b9168f2f921784de84fa445651/raw/penguins.csv'
features = ['bill_length_mm', 'flipper_length_mm']
penguins = pd.read_csv(url)[features + ['species']].dropna()
X_peng = penguins[features].to_numpy()
encoder = LabelEncoder()
y_peng = encoder.fit_transform(penguins['species'])

In [ ]:
#| code-fold: true
xx, yy = np.meshgrid(
    np.linspace(X_peng[:, 0].min()-1, X_peng[:, 0].max()+1, 160),
    np.linspace(X_peng[:, 1].min()-5, X_peng[:, 1].max()+5, 160))
grid_peng = np.column_stack([xx.ravel(), yy.ravel()])
colors = ['#4477AA', '#228833', '#CC6677']
background = ListedColormap(['#dfe8f7', '#e6f5df', '#f8e1df'])

def plot_penguins(m=15):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
    for ax, scale in zip(axes, [False, True]):
        neighbor_model = KNeighborsClassifier(n_neighbors=m)
        model = make_pipeline(StandardScaler(), neighbor_model) if scale else neighbor_model
        model.fit(X_peng, y_peng)
        prediction = model.predict(grid_peng).reshape(xx.shape)
        ax.contourf(xx, yy, prediction, levels=np.arange(4)-0.5, cmap=background)
        for c, species in enumerate(encoder.classes_):
            mask = y_peng == c
            ax.scatter(*X_peng[mask].T, color=colors[c], s=16,
                       edgecolors='white', linewidths=0.3, label=species)
        ax.set(title=f'{"Standardized" if scale else "Original units"}, m = {m}',
               xlabel='Bill length (mm)')
    axes[0].set_ylabel('Flipper length (mm)')
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    plt.show()

plot_penguins(m=15)
if widgets is not None:
    display(widgets.interactive(
        plot_penguins,
        m=widgets.IntSlider(min=1, max=len(X_peng), value=15,
                            continuous_update=False),
    ))


Change $m$ and compare both panels. Decision regions can be nonconvex or disconnected, and their shapes depend on the distance. At $m=N$, every input receives the global majority label. Scaling changes which observations are neighbors even when $m$ stays fixed.

## MNIST: Neighbors Are Images

MNIST contains grayscale images of handwritten digits labeled 0–9. Each image is $28\times28$ pixels; flattening it produces an input vector $x\in\mathbb R^{784}$. The label is the digit shown.

We use a reproducible subset for speed and one train/test split with a fixed $m=5$. First look at the images, then inspect the actual neighbors used for a prediction.


In [ ]:
from sklearn.datasets import fetch_openml

X_mnist, y_mnist = fetch_openml(
    'mnist_784', version=1, as_frame=False, return_X_y=True)
rng = np.random.default_rng(700)
indices = rng.choice(len(X_mnist), size=3000, replace=False)
X_digits = X_mnist[indices].astype(np.float32) / 255.0
y_digits = y_mnist[indices].astype(int)
del X_mnist, y_mnist
X_digit_train, X_digit_test, y_digit_train, y_digit_test = train_test_split(
    X_digits, y_digits, test_size=0.2, stratify=y_digits, random_state=701)

fig, axes = plt.subplots(2, 5, figsize=(8, 3.5))
for ax, digit in zip(axes.flat, range(10)):
    i = np.flatnonzero(y_digit_train == digit)[0]
    ax.imshow(X_digit_train[i].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Label: {digit}')
    ax.axis('off')
fig.tight_layout()
plt.show()

All features are pixel intensities on the same scale. Dividing every pixel by 255 changes distance units but preserves neighbor ordering. We use this simple pixel distance without standardizing each pixel separately.

In [ ]:
digit_model = KNeighborsClassifier(n_neighbors=5).fit(X_digit_train, y_digit_train)
digit_predictions = digit_model.predict(X_digit_test)
print(f'Test accuracy: {accuracy_score(y_digit_test, digit_predictions):.3f}')

query_index = 0
neighbor_indices = digit_model.kneighbors(
    X_digit_test[query_index:query_index+1], return_distance=False)[0]
fig, axes = plt.subplots(1, 6, figsize=(11, 2.5))
axes[0].imshow(X_digit_test[query_index].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
axes[0].set_title(
    f'Query: {y_digit_test[query_index]}\nPrediction: {digit_predictions[query_index]}')
for ax, i in zip(axes[1:], neighbor_indices):
    ax.imshow(X_digit_train[i].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Neighbor: {y_digit_train[i]}')
for ax in axes:
    ax.axis('off')
fig.suptitle('Five nearest neighbors ($m=5$)', y=1.05)
fig.tight_layout()
plt.show()


In [ ]:
#| code-fold: true
wrong = np.flatnonzero(digit_predictions != y_digit_test)[:6]
if len(wrong):
    fig, axes = plt.subplots(1, len(wrong), figsize=(2*len(wrong), 2.5), squeeze=False)
    for ax, i in zip(axes.flat, wrong):
        ax.imshow(X_digit_test[i].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
        ax.set_title(f'True: {y_digit_test[i]}\nPredicted: {digit_predictions[i]}')
        ax.axis('off')
    fig.suptitle('A few test mistakes')
    fig.tight_layout()
    plt.show()

Pixel distance compares corresponding locations. A small shift in handwriting can change many pixel differences even when the digit is still recognizable. Inspect the neighbors and mistakes: does numerical closeness match what you see?

These test results describe the fixed procedure with $m=5$ on this subset. If we use the mistakes to change features or choose $m$, those observations begin to influence model building and a fresh final evaluation is needed.

## Review Questions

See: @sec-knn-questions.
